# Drama Pipeline — Colab GPU worker

Run all to connect your GPU to the private Cloudflare studio. The runtime consumes Colab compute units while allocated. No paid LLM inference API is used.

Before running, add STUDIO_URL, GPU_KEY and your model-download HF_TOKEN in Colab Secrets (key icon in the left sidebar), granting notebook access. Never put credentials in cells or chat.

Initial evaluation: use G4 and Qwen3.6-27B. Choose the smaller Qwen3.5-9B in the last cell if needed. Actual GPU quality/performance, voice calibration and continuity evaluation are still pending.

Finish with **Runtime → Disconnect and delete runtime**. The worker also deletes the runtime after its idle timeout.


In [ ]:
#@title 1. 🔍 Check GPU, system memory and set up SWAP
import os
import sys
import json
import torch
import psutil

!nvidia-smi

if not torch.cuda.is_available():
    raise SystemError("❌ No GPU detected! Enable a GPU via 'Runtime' -> 'Change runtime type'.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
ram_gb = psutil.virtual_memory().total / (1024**3)

print(f"\n{'='*65}")
print(f"🔍 GPU: {gpu_name}")
print(f"🧠 Available VRAM: {vram_gb:.1f} GB")
print(f"💻 System RAM: {ram_gb:.1f} GB")

if "T4" in gpu_name:
    print("\n⚠️ Warning: Tesla T4 detected (15GB VRAM). The MiniMax H3 model is very heavy.")
    print("💡 Creating a 16 GB swap file on the SSD to avoid out-of-memory crashes...")
    !fallocate -l 16G /swapfile && chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile || true
    !swapon --show
    COMFY_EXTRA_ARGS = "--lowvram --disable-async-offload"
elif "L4" in gpu_name:
    print("\n🚀 NVIDIA L4 (24GB VRAM): the models (~44 GB) do not fit at once, ComfyUI will offload to RAM (slower).")
    COMFY_EXTRA_ARGS = "--reserve-vram 2"
elif "RTX PRO 6000" in gpu_name or "Blackwell" in gpu_name:
    print(f"\n🚀 Blackwell GPU! {gpu_name} ({vram_gb:.0f}GB VRAM) - native NVFP4, all models fit in VRAM.")
    COMFY_EXTRA_ARGS = "--reserve-vram 3 --highvram"
elif any(x in gpu_name for x in ["A100", "H100"]):
    print(f"\n🚀 Top-tier performance! {gpu_name} ({vram_gb:.0f}GB VRAM).")
    COMFY_EXTRA_ARGS = "--reserve-vram 3 --highvram"
else:
    COMFY_EXTRA_ARGS = "--reserve-vram 2"

print(f"⚙️ ComfyUI optimization flags: {COMFY_EXTRA_ARGS}")
print(f"{'='*65}\n")


In [ ]:
#@title 2. 📦 Install system utilities and Node.js 24 LTS
import os
import subprocess

print("📦 [1/3] Installing system packages (ffmpeg, aria2, ninja-build)...")
!apt-get update -qq > /dev/null 2>&1; apt-get install -y -qq ffmpeg aria2 ninja-build libgl1 libglib2.0-0 > /tmp/apt-install.log 2>&1 || tail -20 /tmp/apt-install.log
missing = [t for t in ("ffmpeg", "aria2c") if subprocess.run(["which", t], capture_output=True).returncode != 0]
if missing:
    raise RuntimeError(f"❌ System packages failed to install: {missing} (see /tmp/apt-install.log)")

print("📦 [2/3] Installing Node.js 24.x LTS from NodeSource...")
!curl -fsSL https://deb.nodesource.com/setup_24.x | bash - > /dev/null 2>&1
!apt-get install -y -qq nodejs > /dev/null 2>&1

node_ver = !node -v
npm_ver = !npm -v
print(f"✅ Node.js: {node_ver[0] if node_ver else 'OK'} | npm: {npm_ver[0] if npm_ver else 'OK'}")

print("📦 [3/3] Installing Cloudflared and PyNgrok...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!rm -f cloudflared-linux-amd64.deb
!pip install pyngrok psutil --quiet

print("✅ All system utilities installed!")


In [ ]:
#@title 3. 🧩 Install ComfyUI & custom nodes for MiniMax H3
import os

COMFY_DIR = "/content/ComfyUI"

if not os.path.exists(COMFY_DIR):
    print("📥 Cloning ComfyUI...")
    !git clone https://github.com/comfyanonymous/ComfyUI.git $COMFY_DIR
else:
    print("🔄 ComfyUI is already installed. Checking for updates...")
    %cd $COMFY_DIR
    !git pull

%cd $COMFY_DIR
print("📦 Installing ComfyUI Python dependencies...")
!pip install -r requirements.txt --quiet
!pip install triton xformers torchsde soundfile accelerate sentencepiece --quiet

CUSTOM_NODES_DIR = f"{COMFY_DIR}/custom_nodes"
os.makedirs(CUSTOM_NODES_DIR, exist_ok=True)

# Required custom nodes for MiniMax H3
NODES = [
    ("ComfyUI-MiniMax-H3-SPEED", "https://github.com/StanLukuvka/ComfyUI-MiniMax-H3-SPEED.git"),
    ("ComfyUI-PT_H3ConcatAVLatent", "https://github.com/ptmaster/ComfyUI-PT_H3ConcatAVLatent.git"),
    ("Comfyui_Minimax_h3_latent_Upscaler", "https://github.com/LBH-123-AI/Comfyui_Minimax_h3_latent_Upscaler.git"),
    ("ComfyUI-VideoHelperSuite", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),
    ("ComfyUI-KJNodes", "https://github.com/kijai/ComfyUI-KJNodes.git"),
    ("rgthree-comfy", "https://github.com/rgthree/rgthree-comfy.git"),
    ("ComfyUI-GGUF", "https://github.com/city96/ComfyUI-GGUF.git"),
    ("comfyui-minimax-h3-audio-T8", "https://github.com/T8mars/comfyui-minimax-h3-audio-T8.git"),
]

print("\n🧩 Installing custom nodes for MiniMax H3:")
for name, url in NODES:
    target = f"{CUSTOM_NODES_DIR}/{name}"
    if not os.path.exists(target):
        print(f"  + Cloning {name}...")
        !git clone $url $target --quiet
        req_file = f"{target}/requirements.txt"
        if os.path.exists(req_file):
            !pip install -r $req_file --quiet
    else:
        print(f"  ✓ {name} is already installed.")

# Link the output folder to Google Drive
if 'MOUNT_DRIVE' in globals() and MOUNT_DRIVE and os.path.exists(DRIVE_OUTPUT_DIR):
    !rm -rf $COMFY_DIR/output
    !ln -s $DRIVE_OUTPUT_DIR $COMFY_DIR/output
    print(f"\n🔗 ComfyUI/output folder linked to Google Drive!")
else:
    os.makedirs(f"{COMFY_DIR}/output", exist_ok=True)

os.makedirs(f"{COMFY_DIR}/input", exist_ok=True)
print("\n✅ ComfyUI and custom nodes are ready!")


In [ ]:
#@title 4. 🌐 Install MiniMax Studio Web UI (Frontend)
#@markdown MiniMax Studio GitHub repository:
GITHUB_REPO_URL = "https://github.com/hijal/demo-drama-prototype.git" #@param {type:"string"}
BRANCH = "drama-pipeline" #@param {type:"string"}

import os

STUDIO_DIR = "/content/MiniMax_Studio"

if not os.path.exists(STUDIO_DIR):
    print(f"📥 Cloning {GITHUB_REPO_URL} (branch {BRANCH})...")
    !git clone -b $BRANCH $GITHUB_REPO_URL $STUDIO_DIR
elif os.path.exists(STUDIO_DIR) and os.path.exists(f"{STUDIO_DIR}/.git"):
    print("🔄 Updating the Web UI repository...")
    %cd $STUDIO_DIR
    !git pull

if os.path.exists(STUDIO_DIR):
    %cd $STUDIO_DIR

    # Create the ComfyUI_Root wrapper so the SvelteKit API can find ComfyUI/output and ComfyUI/input
    COMFY_WRAPPER = "/content/ComfyUI_Root"
    os.makedirs(COMFY_WRAPPER, exist_ok=True)
    if not os.path.exists(f"{COMFY_WRAPPER}/ComfyUI"):
        !ln -s /content/ComfyUI $COMFY_WRAPPER/ComfyUI

    # Write the .env file
    env_text = f"""COMFY_URL=http://127.0.0.1:8188
COMFY_ROOT={COMFY_WRAPPER}
BIND_HOST=0.0.0.0
PORT=3000
NODE_ENV=production
"""
    with open(f"{STUDIO_DIR}/.env", "w") as f:
        f.write(env_text)
    print("⚙️ .env configuration written.")

    print("📦 Installing npm dependencies (Svelte frontend + ComfyUI API)...\n")
    !npm install --no-audit --no-fund --quiet
    # Production build: dev mode blocks requests coming through the tunnel domain
    print("\n🏗️ Building the production bundle (2-4 min)...")
    !npm run build > /tmp/studio-build.log 2>&1 || (tail -40 /tmp/studio-build.log; exit 1)
    if not os.path.exists(f"{STUDIO_DIR}/web/build/index.js") or not os.path.exists(f"{STUDIO_DIR}/web/dist/index.html"):
        raise RuntimeError("❌ Web UI build failed - see /tmp/studio-build.log")
    print("\n✅ MiniMax Studio Web UI is ready to launch!")
else:
    print(f"ℹ️ Waiting for source files in {STUDIO_DIR}")


In [ ]:
#@title 5. ⏬ Fast download of MiniMax H3 models (aria2c)
#@markdown Tick the models to download (video generation needs all 7):
DOWNLOAD_DIFFUSION = True #@param {type:"boolean"}
#@markdown *Diffusion (FastVideo VSA 1300-step 4-step INT8 ConvRot) — ~22.9 GB*
DOWNLOAD_TEXT_ENCODER = True #@param {type:"boolean"}
#@markdown *Text Encoder (Qwen3VL 32B NVFP4 AWQ) — ~15.7 GB*
DOWNLOAD_VIDEO_VAE = True #@param {type:"boolean"}
#@markdown *Video VAE (INT8 ConvRot) — ~2.6 GB*
DOWNLOAD_AUDIO_VAE = True #@param {type:"boolean"}
#@markdown *Audio VAE (FP32) — ~0.6 GB*
DOWNLOAD_TURBO_LORA = True #@param {type:"boolean"}
#@markdown *Turbo LoRA (4-step v0.1 BF16) — ~1.9 GB*
DOWNLOAD_LATENT_UPSCALER = True #@param {type:"boolean"}
#@markdown *Latent Upscaler 3D (FP16) — ~691 MB*
DOWNLOAD_TAE_PREVIEW = True #@param {type:"boolean"}
#@markdown *TAE Live Preview Decoder — ~9.8 MB*

import os
import sys
import subprocess

MODELS_DIR = "/content/ComfyUI/models"

print("="*65)
print("⏬ CHECKING AND DOWNLOADING MINIMAX H3 MODELS:")
print(f" • Diffusion Model (22.9 GB):    {'ON' if DOWNLOAD_DIFFUSION else 'OFF'}")
print(f" • Text Encoder (15.7 GB):        {'ON' if DOWNLOAD_TEXT_ENCODER else 'OFF'}")
print(f" • Video VAE (2.6 GB):            {'ON' if DOWNLOAD_VIDEO_VAE else 'OFF'}")
print(f" • Audio VAE (0.6 GB):            {'ON' if DOWNLOAD_AUDIO_VAE else 'OFF'}")
print(f" • Turbo LoRA (1.9 GB):           {'ON' if DOWNLOAD_TURBO_LORA else 'OFF'}")
print(f" • Latent Upscaler (0.7 GB):      {'ON' if DOWNLOAD_LATENT_UPSCALER else 'OFF'}")
print(f" • TAE Preview (10 MB):           {'ON' if DOWNLOAD_TAE_PREVIEW else 'OFF'}")
print("="*65 + "\n")

def download_aria2(url, target_dir, filename, min_size_mb=100):
    os.makedirs(target_dir, exist_ok=True)
    full_path = os.path.join(target_dir, filename)
    aria2_file = full_path + ".aria2"
    if os.path.exists(full_path) and not os.path.exists(aria2_file):
        actual_mb = os.path.getsize(full_path) / (1024 * 1024)
        if actual_mb >= min_size_mb:
            print(f"  ✓ {filename} already downloaded ({actual_mb/1024:.2f} GB). Skipping.")
            return
    print(f"\n⏬ Downloading {filename} at high speed...")
    cmd = [
        "aria2c",
        "--console-log-level=warn",
        "-c",
        "-x", "16",
        "-s", "16",
        "-k", "1M",
        "--file-allocation=none",
        "-d", target_dir,
        "-o", filename,
        url
    ]
    subprocess.run(cmd, check=True)
    actual_gb = os.path.getsize(full_path) / (1024**3)
    print(f"  ✅ Done: {filename} ({actual_gb:.2f} GB)")

if DOWNLOAD_DIFFUSION:
    download_aria2(
        "https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/main/minimax_h3_fastvideo_vsa_datafree_1300step_4step_int8_convrot.safetensors",
        f"{MODELS_DIR}/diffusion_models",
        "minimax_h3_fastvideo_vsa_datafree_1300step_4step_int8_convrot.safetensors",
        min_size_mb=20000
    )

if DOWNLOAD_TEXT_ENCODER:
    download_aria2(
        "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
        f"{MODELS_DIR}/text_encoders",
        "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
        min_size_mb=14000
    )

if DOWNLOAD_VIDEO_VAE:
    download_aria2(
        "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_int8_convrot.safetensors",
        f"{MODELS_DIR}/vae",
        "minimax_h3_video_vae_int8_convrot.safetensors",
        min_size_mb=2000
    )

if DOWNLOAD_AUDIO_VAE:
    download_aria2(
        "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors",
        f"{MODELS_DIR}/vae",
        "minimax_h3_audio_vae_fp32.safetensors",
        min_size_mb=500
    )

if DOWNLOAD_TURBO_LORA:
    download_aria2(
        "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",
        f"{MODELS_DIR}/loras",
        "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",
        min_size_mb=1500
    )

if DOWNLOAD_LATENT_UPSCALER:
    download_aria2(
        "https://huggingface.co/LBH-123-AI/Minimax_h3_latent_Upscaler/resolve/main/minimax_h3_latent_upscaler_3d_conv_v1/minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors",
        f"{MODELS_DIR}/latent_upscale_models",
        "minimax_h3_latent_upscaler_3d_fp16.safetensors",
        min_size_mb=600
    )

if DOWNLOAD_TAE_PREVIEW:
    download_aria2(
        "https://huggingface.co/Kijai/MiniMax-H3-TAE/resolve/main/vae_approx/taeh3.safetensors",
        f"{MODELS_DIR}/vae_approx",
        "taeh3.safetensors",
        min_size_mb=8
    )

print("\n🎉 All selected models downloaded and ready!")


In [ ]:
#@title 6. Start the pipeline GPU worker (uses compute units)
import os
import shlex
import subprocess
import sys
import time
import urllib.request
import urllib.parse
import json
from pathlib import Path
from google.colab import userdata, runtime

# Same look as the GPU worker's log lines: time, icon, message.
STARTED = time.time()
def say(icon, text, colour="36"):
    print(f"\x1b[2m{time.strftime('%H:%M:%S')}\x1b[0m \x1b[{colour}m{icon}\x1b[0m {text}", flush=True)
def step(text): say("›", text)
def done(text): say("✓", f"{text} \x1b[2m({time.time() - STARTED:.0f}s)\x1b[0m", "32")
def show_tail(name, path, lines=20):
    try:
        content = Path(path).read_text(errors="replace").rstrip().splitlines()[-lines:]
    except OSError:
        return
    say("✗", f"{name} log, last {len(content)} lines ({path}):", "31")
    for line in content:
        print(f"\x1b[2m           │ {line[:200]}\x1b[0m")

COLAB_LLM_MODEL = "Qwen/Qwen3.6-27B" #@param ["Qwen/Qwen3.6-27B", "Qwen/Qwen3.5-9B"]
CONTINUITY_STRATEGY = "reference-video" #@param ["reference-video", "latent"]
# Latent mode is available for GPU evaluation; no quality winner is selected yet.
IDLE_MINUTES = 20 #@param {type:"integer"}
RENDER_QUALITY = "1080p" #@param ["1080p", "720p"]
# 720p renders faster for testing the pipeline; final films use 1080p.
QA_MIN_SCORE = 75 #@param {type:"slider", min:50, max:95, step:5}
# QA approves a reference or scene at this score (%); below it, it retries up to 3 times and keeps the best.
TIMEZONE = "Asia/Dhaka" #@param {type:"string"}
# Log times use this zone; Colab itself runs on UTC.
os.environ["TZ"] = TIMEZONE
time.tzset()
VOICE_MIN_SIMILARITY = "" #@param {type:"string"}
# Leave the threshold empty until calibrated against approved voice samples.
# Uncalibrated voices need owner review, without automatic retakes.
for name in ["STUDIO_URL", "GPU_KEY", "HF_TOKEN"]:
    try:
        os.environ[name] = userdata.get(name)
    except Exception:
        if name != "HF_TOKEN":
            raise RuntimeError(f"Add {name} in Colab Secrets and allow notebook access")
COMFY_LOG = "/tmp/pipeline-comfy.log"
MODEL_LOG = "/tmp/pipeline-model.log"
os.environ["COMFY_LOG_FILE"] = COMFY_LOG
os.environ["MODEL_LOG_FILE"] = MODEL_LOG
os.environ["PIPELINE_WORK_DIR"] = "/content/drama-runtime"
os.environ["COMFY_OUTPUT_DIR"] = "/content/drama-comfy-output"
os.environ["COMFY_INPUT_DIR"] = "/content/drama-comfy-input"
for directory in [os.environ["COMFY_OUTPUT_DIR"], os.environ["COMFY_INPUT_DIR"]]:
    Path(directory).mkdir(parents=True, exist_ok=True)
os.environ["COLAB_LLM_MODEL"] = COLAB_LLM_MODEL
os.environ["IDLE_MINUTES"] = str(IDLE_MINUTES)
os.environ["CONTINUITY_STRATEGY"] = CONTINUITY_STRATEGY
os.environ["RENDER_SHORT_SIDE"] = RENDER_QUALITY.rstrip("p")
os.environ["QA_MIN_SCORE"] = str(QA_MIN_SCORE)
# Validate the remote GPU key/bypass before installing the second model environment.
class NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, request, fp, code, msg, headers, newurl):
        return None
step("Checking the studio connection")
studio_url = os.environ["STUDIO_URL"].rstrip("/")
parsed_url = urllib.parse.urlsplit(studio_url)
if parsed_url.scheme != "https" or not parsed_url.netloc or parsed_url.path or parsed_url.query or parsed_url.fragment or parsed_url.username:
    raise RuntimeError("STUDIO_URL must be the HTTPS studio origin without a path")
try:
    request = urllib.request.Request(studio_url + "/api/gpu/health", data=b"{}", headers={"Content-Type": "application/json", "X-GPU-Key": os.environ["GPU_KEY"], "User-Agent": "Drama-Pipeline-Colab"})
    with urllib.request.build_opener(NoRedirect()).open(request, timeout=30) as response:
        health = json.load(response)
    if health.get("protocol", 0) < 2:
        raise RuntimeError("Deploy the current pipeline backend before running this notebook")
except Exception:
    raise RuntimeError("Studio GPU preflight failed. Check STUDIO_URL, GPU_KEY and the /api/gpu/* Access bypass. Credentials are not printed.") from None
done(f"Studio reachable: {parsed_url.netloc}")
if VOICE_MIN_SIMILARITY.strip():
    float(VOICE_MIN_SIMILARITY)
    os.environ["VOICE_MIN_SIMILARITY"] = VOICE_MIN_SIMILARITY
if sys.version_info < (3, 11):
    raise RuntimeError("The local model environment requires Python 3.11 or newer")
step("Installing the model environment (Qwen, FLUX.2 Klein, Whisper); a few minutes on first run")
venv = "/content/drama-model-env"
# Colab's Python has no python3.X-venv package (ensurepip is missing), so stdlib venv fails; virtualenv bundles pip.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "virtualenv"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", "--quiet", "--system-site-packages", venv], check=True)
python = f"{venv}/bin/python"
# Quiet pip keeps the cell readable; the full output is shown only when the install fails.
install = subprocess.run([python, "-m", "pip", "install", "--quiet", "--upgrade", "-r", f"{STUDIO_DIR}/gpu/python/requirements.txt"], capture_output=True, text=True)
if install.returncode != 0:
    print(install.stdout[-4000:], install.stderr[-4000:])
    raise RuntimeError("Model environment install failed; see the pip output above")
with open("/content/drama-model-versions.txt", "w") as versions:
    subprocess.run([python, "-m", "pip", "freeze"], check=True, stdout=versions)
done("Model environment ready")
# Keep the original notebook's H3 flags, with argument-list execution.
flags = shlex.split(COMFY_EXTRA_ARGS if "COMFY_EXTRA_ARGS" in globals() else "--reserve-vram 2")
comfy = model_server = worker = None

def wait_for(name, url, process, log_path):
    for _ in range(180):
        if process.poll() is not None:
            show_tail(name, log_path)
            raise RuntimeError(f"{name} stopped while starting; see its log above")
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                if response.status == 200:
                    return
        except Exception:
            time.sleep(2)
    show_tail(name, log_path)
    raise RuntimeError(f"{name} did not start within 6 minutes; see its log above")

if CONTINUITY_STRATEGY == "latent":
    node_dir = Path(COMFY_DIR) / "custom_nodes" / "ComfyUI-H3-Motion-Context"
    revision = "518c083d11d80fa5920222ad89889924ee4401c9"
    if not node_dir.exists():
        subprocess.run(["git", "clone", "--no-checkout", "https://github.com/NikoDemon80/ComfyUI-H3-Motion-Context.git", str(node_dir)], check=True)
    subprocess.run(["git", "-C", str(node_dir), "fetch", "--depth", "1", "origin", revision], check=True)
    subprocess.run(["git", "-C", str(node_dir), "checkout", "--detach", revision], check=True)

try:
    step("Starting ComfyUI and the local model server")
    comfy = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch", "--output-directory", os.environ["COMFY_OUTPUT_DIR"], "--input-directory", os.environ["COMFY_INPUT_DIR"], "--max-upload-size", "1024", *flags], cwd=COMFY_DIR, stdout=open(COMFY_LOG, "w"), stderr=subprocess.STDOUT)
    model_server = subprocess.Popen([python, f"{STUDIO_DIR}/gpu/python/model_server.py"], cwd=STUDIO_DIR, stdout=open(MODEL_LOG, "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
    wait_for("ComfyUI", "http://127.0.0.1:8188/system_stats", comfy, COMFY_LOG)
    done("ComfyUI ready")
    wait_for("Model server", "http://127.0.0.1:8000/health", model_server, MODEL_LOG)
    done("Model server ready")
    if CONTINUITY_STRATEGY == "latent":
        for node in ["MiniMaxH3MotionContext", "MiniMaxH3MotionContextTrim", "MiniMaxH3MotionContextSaveLatent", "MiniMaxH3MotionContextLoadLatent"]:
            with urllib.request.urlopen("http://127.0.0.1:8188/object_info/" + node, timeout=30) as response:
                if node not in json.load(response):
                    show_tail("ComfyUI", COMFY_LOG)
                    raise RuntimeError("Motion Context nodes failed to load; see the ComfyUI log above")
    say("●", f"GPU worker running. Open {studio_url} to work on your project. Compute units are being used.", "33")
    worker = subprocess.Popen(["npm", "--workspace", "gpu", "start"], cwd=STUDIO_DIR, env=os.environ.copy())
    if worker.wait() != 0:
        raise RuntimeError("GPU worker stopped with an error; see the red lines above")
    say("■", "Worker finished. Deleting the runtime to stop compute-unit usage.", "33")
    runtime.unassign()
finally:
    for process in [worker, model_server, comfy]:
        if process is not None and process.poll() is None:
            process.terminate()
